In [1]:
from IPython.display import Video, display
import cv2
import numpy as np

from utils.color import detect_color
from utils.ROI import roi, get_overlap_componant


In [16]:
in_path  = "../data/Video1fixed.mp4"       
out_path = "../outputs/group_11_fixed_test.mp4"

cap = cv2.VideoCapture(in_path)
if not cap.isOpened():
    raise IOError(f"Could not open {in_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v") 
writer = cv2.VideoWriter(out_path, fourcc, fps, (w, h))

ok, first_frame = cap.read()
if not ok:
    print("no frames")

while True:
    ok, frame_bgr = cap.read()
    if not ok:
        break

    green = [0,255,0]
    green_mask = detect_color(frame_bgr, green, [40,40], [255,255], tuning=35)

    blue = [0,0,255]
    blue_mask = detect_color(frame_bgr, blue, [100,100], [255,255], tuning=25)
    ball_mask = roi(blue_mask, s_erod=3, s_dil=10)

    is_overlapping, cloak_mask = get_overlap_componant(green_mask, ball_mask)

    if is_overlapping:
        print("YEPSSSS")
        visible_environment_mask = cv2.bitwise_not(cloak_mask)

        invisible_cloak = cv2.bitwise_and(first_frame, first_frame, mask=cloak_mask)
        visible_environment = cv2.bitwise_and(frame_bgr, frame_bgr, mask=visible_environment_mask)

        output_frame = cv2.add(visible_environment, invisible_cloak)
    else:
        output_frame = frame_bgr
    
    masking = ball_mask + green_mask
    frame = cv2.bitwise_and(frame_bgr, frame_bgr, mask=masking)

    writer.write(frame)

    #writer.write(output_frame)

cap.release()
writer.release()
print("Saved:", out_path)

display(Video(out_path))

Saved: ../outputs/group_11_fixed_test.mp4
